# FinGPT Resume Backtest: Track A (CPU-only)

This notebook is the lightweight recovery path for an existing `backtest_*.csv`.

What Track A does:
- load an existing backtest CSV
- re-apply Agent 2 PMI post-processing with a configurable `PMI_ALPHA`
- fetch realized returns with slow, single-threaded, throttled requests
- recompute metrics and save a new CSV

What Track A does not do:
- it does not re-run Agent 1
- it does not re-run Agent 2 model inference
- it is designed to run on Colab CPU


In [ ]:
# Cell 1 - Install lightweight dependencies
import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

_pip("yfinance", "pandas", "numpy", "requests")
print("Dependencies installed.")


In [ ]:
# Cell 2 - Paths and knobs
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RESUME_CSV = "/content/drive/MyDrive/backtest_20260506T051302Z.csv"
OUT_DIR = "/content/drive/MyDrive"
REPO_DIR = "/content/drive/MyDrive/FinGPT_Part2"
PMI_PRIOR_PATH = os.path.join(REPO_DIR, "output/pmi_null_logprobs.json")

PMI_ALPHA = 1.0
CALIBRATION_T = 1.2
PRICE_FETCH_RETRIES = 3
PRICE_FETCH_SLEEP_SEC = 0.35
PRICE_FETCH_BATCH_SIZE = 20
PRICE_FETCH_BATCH_PAUSE_SEC = 2.0
REFRESH_EXISTING_PRICES = True

os.makedirs(OUT_DIR, exist_ok=True)
print(f"Resume CSV    : {RESUME_CSV}")
print(f"Output dir    : {OUT_DIR}")
print(f"PMI prior path: {PMI_PRIOR_PATH}")
print(f"PMI alpha     : {PMI_ALPHA}")


In [ ]:
# Cell 3 - Load current backtest CSV and validate logits columns
import numpy as np
import pandas as pd

raw_df = pd.read_csv(RESUME_CSV)
print(f"Loaded {len(raw_df)} rows.")
print(f"Columns: {list(raw_df.columns)}")

required_raw_cols = [
    "raw_signal_logprob_A", "raw_signal_logprob_B", "raw_signal_logprob_C",
]
missing = [c for c in required_raw_cols if c not in raw_df.columns]
if missing:
    raise ValueError(f"Resume CSV is missing required raw logprob columns: {missing}")

raw_df["logits_list"] = raw_df.apply(
    lambda row: [row["raw_signal_logprob_A"], row["raw_signal_logprob_B"], row["raw_signal_logprob_C"]],
    axis=1,
)
has_logits = raw_df[required_raw_cols].notna().all(axis=1)
has_row_null = all(
    col in raw_df.columns for col in ["pmi_null_logprob_A", "pmi_null_logprob_B", "pmi_null_logprob_C"]
)

print(f"Rows with valid A/B/C logits : {has_logits.sum()} / {len(raw_df)}")
print(f"Rows without logits          : {(~has_logits).sum()} / {len(raw_df)}")
print(f"Per-row PMI null logprobs present: {has_row_null}")
print()
print("Original signal_direction distribution:")
if "signal_direction" in raw_df.columns:
    print(raw_df["signal_direction"].value_counts(dropna=False).to_string())
else:
    print("signal_direction column not found.")


In [ ]:
# Cell 4 - Offline PMI re-scoring with configurable alpha
import json

STRATEGY_SET = ["BUY", "HOLD", "SELL"]
_DIR_MAP = {0: "long", 1: "neutral", 2: "short"}
_POS_MAP = {"long": 1, "short": -1, "neutral": 0}

def _softmax(x, T=CALIBRATION_T):
    x = np.array(x, dtype=float) / T
    x -= x.max()
    e = np.exp(x)
    return e / e.sum()

def _load_pmi_prior(path: str):
    if not path or not os.path.exists(path):
        return None
    try:
        with open(path, encoding="utf-8") as fh:
            data = json.load(fh)
        lp = data.get("null_logprobs")
        if isinstance(lp, list) and len(lp) == 3:
            return np.array(lp, dtype=float), data
    except Exception as exc:
        print(f"Warning: could not read PMI cache ({exc})")
    return None

logits_arr = np.array(raw_df.loc[has_logits, "logits_list"].tolist(), dtype=float)
_disk = _load_pmi_prior(PMI_PRIOR_PATH)

if has_row_null:
    null_arr = raw_df.loc[has_logits, ["pmi_null_logprob_A", "pmi_null_logprob_B", "pmi_null_logprob_C"]].to_numpy(dtype=float)
    pmi_source = "per-row pmi_null_logprob_* columns from CSV"
    print(f"PMI prior source : {pmi_source}")
elif _disk is not None:
    null_lp, _meta = _disk
    null_arr = np.repeat(null_lp.reshape(1, 3), len(logits_arr), axis=0)
    pmi_source = "disk cache (true model prior)"
    print(f"PMI prior source : {pmi_source}")
    print(f"  path           : {PMI_PRIOR_PATH}")
    print(f"  decision_prefix: {_meta.get('decision_prefix')}")
    print(f"  score_tokens   : {_meta.get('score_tokens')}")
    print(f"  computed_at    : {_meta.get('computed_at')}")
else:
    null_lp = logits_arr.mean(axis=0)
    null_arr = np.repeat(null_lp.reshape(1, 3), len(logits_arr), axis=0)
    pmi_source = "empirical mean (fallback)"
    print(f"PMI prior source : {pmi_source}")

print(f"PMI alpha = {PMI_ALPHA}")
pmi_arr = logits_arr - PMI_ALPHA * null_arr

corrected = raw_df.copy()
new_rows = []
for df_idx, logits_pmi in zip(raw_df.index[has_logits], pmi_arr):
    probs = _softmax(logits_pmi)
    best = int(probs.argmax())
    direction = _DIR_MAP[best]
    confidence = round(float(probs[best]), 6)
    realized = raw_df.loc[df_idx, "realized_return"] if "realized_return" in raw_df.columns else np.nan
    realized = float(realized) if pd.notna(realized) else np.nan
    position = _POS_MAP[direction] if pd.notna(realized) else np.nan
    strategy_return = position * realized if pd.notna(realized) else np.nan
    new_rows.append({
        "_df_idx": df_idx,
        "direction": direction,
        "confidence": confidence,
        "signal_direction": direction,
        "signal_confidence": confidence,
        "signal_prob_A": round(float(probs[0]), 6),
        "signal_prob_B": round(float(probs[1]), 6),
        "signal_prob_C": round(float(probs[2]), 6),
        "pmi_adjusted_logit_A": float(logits_pmi[0]),
        "pmi_adjusted_logit_B": float(logits_pmi[1]),
        "pmi_adjusted_logit_C": float(logits_pmi[2]),
        "pmi_alpha_used": PMI_ALPHA,
        "position": position,
        "strategy_return": strategy_return,
    })

pmi_df = pd.DataFrame(new_rows).set_index("_df_idx")
for col in pmi_df.columns:
    corrected.loc[has_logits, col] = pmi_df[col]

print("PMI-corrected signal_direction distribution:")
print(corrected["signal_direction"].value_counts(dropna=False).to_string())
print("PMI-corrected direction distribution:")
print(corrected["direction"].value_counts(dropna=False).to_string())


In [ ]:
# Cell 5 - Fetch realized returns with retry + throttling
import time
import yfinance as yf

_price_cache = {}

def _fetch_return(ticker: str, start_date: str, end_date: str):
    key = (ticker, start_date, end_date)
    if key in _price_cache:
        return _price_cache[key]
    last_error = ""
    for attempt in range(PRICE_FETCH_RETRIES):
        try:
            hist = yf.download(
                tickers=ticker,
                start=start_date,
                end=end_date,
                interval="1d",
                auto_adjust=True,
                progress=False,
                threads=False,
            )
            if hist is None or hist.empty:
                last_error = "empty_history"
            else:
                if isinstance(hist.columns, pd.MultiIndex):
                    hist.columns = hist.columns.get_level_values(0)
                if "Close" not in hist.columns:
                    last_error = "missing_close"
                else:
                    closes = hist["Close"].dropna()
                    if closes.empty:
                        last_error = "empty_closes"
                    else:
                        first, last = float(closes.iloc[0]), float(closes.iloc[-1])
                        if first == 0.0:
                            last_error = "zero_first_close"
                        else:
                            ret = (last - first) / first
                            _price_cache[key] = (ret, "")
                            return _price_cache[key]
        except Exception as exc:
            last_error = f"exception: {exc}"
        time.sleep(PRICE_FETCH_SLEEP_SEC * (attempt + 1))
    _price_cache[key] = (None, last_error or "unknown")
    return _price_cache[key]

active = corrected[corrected["signal_direction"].notna()].copy()
if not REFRESH_EXISTING_PRICES and "realized_return" in active.columns:
    already_have_price = active["realized_return"].notna()
else:
    already_have_price = pd.Series([False] * len(active), index=active.index)

realized_returns = []
price_error_reasons = []
fetch_ok = 0
for i, (_, row) in enumerate(active.iterrows()):
    if already_have_price.iloc[i]:
        realized_returns.append(float(row["realized_return"]))
        price_error_reasons.append("")
        fetch_ok += 1
    else:
        ret, reason = _fetch_return(str(row["ticker"]), str(row["start_date"]), str(row["end_date"]))
        realized_returns.append(ret)
        price_error_reasons.append(reason)
        if ret is not None:
            fetch_ok += 1
    if (i + 1) % 50 == 0:
        print(f"  price fetch {i+1}/{len(active)}  ok={fetch_ok}")
    if (i + 1) % PRICE_FETCH_BATCH_SIZE == 0:
        print(f"  pausing {PRICE_FETCH_BATCH_PAUSE_SEC}s to avoid rate-limit bursts...")
        time.sleep(PRICE_FETCH_BATCH_PAUSE_SEC)

print(f"Price fetch done: {fetch_ok}/{len(active)} valid returns.")


In [ ]:
# Cell 6 - Assemble final DataFrame and compute metrics
import math

_POS_MAP = {"long": 1, "short": -1, "neutral": 0}

def _direction_from_return(r, threshold=0.001):
    if r > threshold:
        return "up"
    if r < -threshold:
        return "down"
    return "neutral"

active = active.copy()
active["realized_return"] = realized_returns
active["price_fetch_error_reason"] = price_error_reasons
active["position"] = active["signal_direction"].map(_POS_MAP).fillna(0).astype(int)
active["strategy_return"] = [
    _POS_MAP.get(row["signal_direction"], 0) * row["realized_return"]
    if row["realized_return"] is not None and pd.notna(row["realized_return"])
    else np.nan
    for _, row in active.iterrows()
]
active["skipped_reason"] = [
    "" if r is not None and pd.notna(r) else "price_fetch_failed"
    for r in realized_returns
]

no_signal = corrected[corrected["signal_direction"].isna()].copy()
no_signal["skipped_reason"] = no_signal.get("skipped_reason", pd.Series(index=no_signal.index)).fillna("signal_failed")
if "price_fetch_error_reason" not in no_signal.columns:
    no_signal["price_fetch_error_reason"] = ""

final = pd.concat([active, no_signal], ignore_index=True)
successful = final[final["skipped_reason"] == ""].copy()
total = len(final)
ok = len(successful)
skipped = total - ok

if ok > 0:
    successful["realized_direction"] = successful["realized_return"].astype(float).apply(_direction_from_return)
    successful["signal_mkt_dir"] = successful["signal_direction"].map({"long": "up", "short": "down", "neutral": "neutral"})
    dir_acc = float((successful["signal_mkt_dir"] == successful["realized_direction"]).mean())
    longs = successful[successful["signal_direction"] == "long"]
    shorts = successful[successful["signal_direction"] == "short"]
    long_acc = float((longs["realized_direction"] == "up").mean()) if len(longs) else float("nan")
    short_acc = float((shorts["realized_direction"] == "down").mean()) if len(shorts) else float("nan")
    sr = successful["strategy_return"].astype(float)
    mu = float(sr.mean())
    std = float(sr.std(ddof=0))
    sharpe = (mu / std) * math.sqrt(52) if std > 0 else float("nan")
    total_pnl = float(sr.sum())
    fingpt_dir = successful["fingpt_label"].map({"up": "long", "down": "short", "neutral": "neutral"})
    vs_fingpt = float((successful["signal_direction"] == fingpt_dir).mean())
else:
    dir_acc = long_acc = short_acc = sharpe = total_pnl = vs_fingpt = float("nan")
    mu = std = 0.0

metrics = {
    "total_rows": total,
    "successful_rows": ok,
    "skip_rate": (skipped / total) if total else 0.0,
    "direction_accuracy": dir_acc,
    "long_accuracy": long_acc,
    "short_accuracy": short_acc,
    "mean_strategy_return": mu,
    "std_strategy_return": std,
    "annualized_sharpe": sharpe,
    "total_pnl": total_pnl,
    "vs_fingpt_accuracy": vs_fingpt,
}

print("=" * 50)
print("BACKTEST METRICS (PMI-corrected signals)")
print("=" * 50)
for k, v in metrics.items():
    print(f"  {k:<28}: {v:.4f}" if isinstance(v, float) else f"  {k:<28}: {v}")
print()
print("Signal direction breakdown (successful rows):")
if ok > 0:
    print(successful["signal_direction"].value_counts(dropna=False).to_string())
print()
print("Top price fetch failures:")
print(final["price_fetch_error_reason"].fillna("").replace("", "<ok>").value_counts().head(10).to_string())


In [ ]:
# Cell 7 - Save corrected CSV
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
alpha_slug = str(PMI_ALPHA).replace(".", "_").replace("-", "neg_")
out_path = os.path.join(OUT_DIR, f"backtest_pmi_alpha_{alpha_slug}_{timestamp}.csv")
final.drop(columns=["logits_list"], errors="ignore").to_csv(out_path, index=False)
print(f"Saved {len(final)} rows to {out_path}")


## Grid Search And Visualization

下面这部分会基于当前 `final` DataFrame 做两套策略研究：
- Mixed strategy: long / neutral / short
- Long-only strategy: 只在 BUY 满足阈值时持有，否则空仓


In [ ]:
# Cell 8 - Multi-dimensional grid search for mixed + long-only
from backtest.pmi_grid_search import (
    apply_long_only_grid_strategy,
    apply_pmi_alpha_to_results,
    run_no_pmi_confidence_margin_grid_search,
    run_no_pmi_long_only_grid_search,
    run_alpha_confidence_margin_grid_search,
    run_long_only_grid_search,
)

ALPHA_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]
CONFIDENCE_GRID = [0.30, 0.35, 0.40, 0.45, 0.50]
MARGIN_GRID = [0.00, 0.05, 0.10, 0.15, 0.20]
GRID_CALIBRATION_T = CALIBRATION_T

GRID_DIR = os.path.join(OUT_DIR, 'grid_search_outputs')
os.makedirs(GRID_DIR, exist_ok=True)

mixed_summary_path = os.path.join(GRID_DIR, 'mixed_alpha_confidence_margin_grid.csv')
long_only_summary_path = os.path.join(GRID_DIR, 'long_only_alpha_confidence_margin_grid.csv')
no_pmi_mixed_summary_path = os.path.join(GRID_DIR, 'no_pmi_mixed_confidence_margin_grid.csv')
no_pmi_long_only_summary_path = os.path.join(GRID_DIR, 'no_pmi_long_only_confidence_margin_grid.csv')

mixed_summary = run_alpha_confidence_margin_grid_search(
    final,
    alphas=ALPHA_GRID,
    confidence_levels=CONFIDENCE_GRID,
    margin_levels=MARGIN_GRID,
    output_path=mixed_summary_path,
    calibration_t=GRID_CALIBRATION_T,
)
long_only_summary = run_long_only_grid_search(
    final,
    alphas=ALPHA_GRID,
    confidence_levels=CONFIDENCE_GRID,
    margin_levels=MARGIN_GRID,
    output_path=long_only_summary_path,
    calibration_t=GRID_CALIBRATION_T,
)
no_pmi_mixed_summary = run_no_pmi_confidence_margin_grid_search(
    final,
    confidence_levels=CONFIDENCE_GRID,
    margin_levels=MARGIN_GRID,
    output_path=no_pmi_mixed_summary_path,
    calibration_t=GRID_CALIBRATION_T,
)
no_pmi_long_only_summary = run_no_pmi_long_only_grid_search(
    final,
    confidence_levels=CONFIDENCE_GRID,
    margin_levels=MARGIN_GRID,
    output_path=no_pmi_long_only_summary_path,
    calibration_t=GRID_CALIBRATION_T,
)

print(f'Mixed summary saved to: {mixed_summary_path}')
print(f'Long-only summary saved to: {long_only_summary_path}')
print(f'No-PMI mixed summary saved to: {no_pmi_mixed_summary_path}')
print(f'No-PMI long-only summary saved to: {no_pmi_long_only_summary_path}')
print('\nMixed grid top 10 by total_pnl:')
display(mixed_summary.sort_values(['total_pnl', 'annualized_sharpe'], ascending=False).head(10))
print('\nLong-only grid top 10 by total_pnl:')
display(long_only_summary.sort_values(['total_pnl', 'annualized_sharpe'], ascending=False).head(10))
print('\nNo-PMI mixed grid top 10 by total_pnl:')
display(no_pmi_mixed_summary.sort_values(['total_pnl', 'annualized_sharpe'], ascending=False).head(10))
print('\nNo-PMI long-only grid top 10 by total_pnl:')
display(no_pmi_long_only_summary.sort_values(['total_pnl', 'annualized_sharpe'], ascending=False).head(10))


In [ ]:
# Cell 9 - Materialize the best mixed strategy and best long-only strategy
def _pick_best(summary_df):
    ranked = summary_df.sort_values(
        ['total_pnl', 'annualized_sharpe', 'direction_accuracy'],
        ascending=False,
    ).reset_index(drop=True)
    return ranked.iloc[0]

best_mixed = _pick_best(mixed_summary)
best_long_only = _pick_best(long_only_summary)

best_mixed_df = apply_pmi_alpha_to_results(
    final,
    pmi_alpha=float(best_mixed['pmi_alpha']),
    calibration_t=float(best_mixed['calibration_T']),
    min_confidence=float(best_mixed['signal_min_confidence']),
    min_margin=float(best_mixed['signal_min_margin']),
)

best_long_only_df = apply_long_only_grid_strategy(
    final,
    pmi_alpha=float(best_long_only['pmi_alpha']),
    confidence_threshold=float(best_long_only['signal_min_confidence']),
    margin_threshold=float(best_long_only['signal_min_margin']),
    calibration_t=float(best_long_only['calibration_T']),
)

best_mixed_path = os.path.join(GRID_DIR, 'best_mixed_strategy.csv')
best_long_only_path = os.path.join(GRID_DIR, 'best_long_only_strategy.csv')
best_mixed_df.to_csv(best_mixed_path, index=False)
best_long_only_df.to_csv(best_long_only_path, index=False)

print('Best mixed params:')
display(best_mixed.to_frame().T)
print('Best long-only params:')
display(best_long_only.to_frame().T)
print(f'Best mixed CSV saved to: {best_mixed_path}')
print(f'Best long-only CSV saved to: {best_long_only_path}')


In [ ]:
# Cell 10 - Visualization helpers
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROLLING_WINDOW = 12

def _successful_rows(df):
    if 'skipped_reason' not in df.columns:
        out = df.copy()
    else:
        out = df[df['skipped_reason'].fillna('') == ''].copy()
    if 'end_date' in out.columns:
        out['end_date'] = pd.to_datetime(out['end_date'], errors='coerce')
        out = out.sort_values(['end_date', 'ticker'], kind='stable')
    return out.reset_index(drop=True)

def _equity_curve(df):
    returns = df['strategy_return'].fillna(0.0).astype(float)
    equity = (1.0 + returns).cumprod()
    drawdown = equity / equity.cummax() - 1.0
    return returns, equity, drawdown

def _rolling_stats(df, window=ROLLING_WINDOW):
    returns = df['strategy_return'].fillna(0.0).astype(float)
    win_rate = (returns > 0).astype(float).rolling(window, min_periods=1).mean()
    rolling_return = returns.rolling(window, min_periods=1).mean()
    return win_rate, rolling_return

def _signal_label_series(df):
    mapping = {'long': 'BUY', 'neutral': 'HOLD', 'short': 'SELL'}
    return df['signal_direction'].map(mapping).fillna('UNKNOWN')

def _signal_avg_returns(df):
    out = df.copy()
    out['signal_bucket'] = _signal_label_series(out)
    grouped = out.groupby('signal_bucket')['realized_return'].mean()
    ordered = pd.Series(index=['BUY', 'SELL', 'HOLD'], dtype=float)
    for key in ordered.index:
        ordered.loc[key] = grouped.get(key, np.nan)
    return ordered

def _plot_strategy_dashboard(df, strategy_name):
    ok = _successful_rows(df)
    returns, equity, drawdown = _equity_curve(ok)
    rolling_win_rate, rolling_return = _rolling_stats(ok)
    avg_signal_returns = _signal_avg_returns(ok)
    x = ok['end_date'] if 'end_date' in ok.columns else ok.index

    fig, axes = plt.subplots(3, 2, figsize=(16, 16))
    axes = axes.ravel()

    axes[0].plot(x, equity, color='navy', linewidth=2)
    axes[0].set_title(f'{strategy_name} - Cumulative Return / Equity Curve')
    axes[0].set_ylabel('Equity')

    axes[1].fill_between(x, drawdown, 0, color='firebrick', alpha=0.35)
    axes[1].set_title(f'{strategy_name} - Drawdown Curve')
    axes[1].set_ylabel('Drawdown')

    axes[2].hist(returns.dropna(), bins=30, color='darkslateblue', alpha=0.8)
    axes[2].set_title(f'{strategy_name} - Histogram of Trade/Period Returns')
    axes[2].set_xlabel('Strategy return')

    axes[3].plot(x, rolling_win_rate, color='darkgreen', linewidth=2)
    axes[3].set_title(f'{strategy_name} - Rolling Win Rate ({ROLLING_WINDOW})')
    axes[3].set_ylim(0, 1)

    axes[4].plot(x, rolling_return, color='darkorange', linewidth=2)
    axes[4].set_title(f'{strategy_name} - Rolling Return ({ROLLING_WINDOW})')
    axes[4].axhline(0, color='black', linewidth=1, alpha=0.5)

    axes[5].bar(avg_signal_returns.index, avg_signal_returns.values, color=['seagreen', 'indianred', 'gray'])
    axes[5].set_title(f'{strategy_name} - Average Return By Signal')
    axes[5].set_ylabel('Average realized return')
    axes[5].axhline(0, color='black', linewidth=1, alpha=0.5)

    for ax in axes[:5]:
        ax.grid(alpha=0.2)

    plt.tight_layout()
    return fig, ok


In [ ]:
# Cell 11 - Plot best mixed and best long-only strategies
mixed_fig, mixed_ok = _plot_strategy_dashboard(best_mixed_df, 'Mixed Strategy')
long_only_fig, long_only_ok = _plot_strategy_dashboard(best_long_only_df, 'Long-only Strategy')
plt.show()

mixed_plot_path = os.path.join(GRID_DIR, 'mixed_strategy_dashboard.png')
long_only_plot_path = os.path.join(GRID_DIR, 'long_only_strategy_dashboard.png')
mixed_fig.savefig(mixed_plot_path, dpi=160, bbox_inches='tight')
long_only_fig.savefig(long_only_plot_path, dpi=160, bbox_inches='tight')
print(f'Saved mixed dashboard to: {mixed_plot_path}')
print(f'Saved long-only dashboard to: {long_only_plot_path}')


In [ ]:
# Cell 12 - Visualize grid-search surfaces and signal average returns side by side
import matplotlib.pyplot as plt

def _plot_grid_heatmap(summary_df, strategy_name, metric='total_pnl'):
    best_alpha = (
        summary_df.groupby('pmi_alpha')[metric]
        .max()
        .sort_values(ascending=False)
        .index[0]
    )
    subset = summary_df[summary_df['pmi_alpha'] == best_alpha].copy()
    pivot = subset.pivot_table(
        index='signal_min_margin',
        columns='signal_min_confidence',
        values=metric,
        aggfunc='mean',
    ).sort_index().sort_index(axis=1)
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(pivot.values, aspect='auto', origin='lower', cmap='viridis')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{v:.2f}' for v in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{v:.2f}' for v in pivot.index])
    ax.set_xlabel('Confidence threshold')
    ax.set_ylabel('Margin threshold')
    ax.set_title(f'{strategy_name} - {metric} heatmap (best alpha={best_alpha:.2f})')
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    return fig

mixed_heatmap_fig = _plot_grid_heatmap(mixed_summary, 'Mixed Strategy', metric='total_pnl')
long_only_heatmap_fig = _plot_grid_heatmap(long_only_summary, 'Long-only Strategy', metric='total_pnl')
plt.show()

mixed_heatmap_fig.savefig(os.path.join(GRID_DIR, 'mixed_total_pnl_heatmap.png'), dpi=160, bbox_inches='tight')
long_only_heatmap_fig.savefig(os.path.join(GRID_DIR, 'long_only_total_pnl_heatmap.png'), dpi=160, bbox_inches='tight')

print('Mixed strategy average returns by signal:')
display(_signal_avg_returns(mixed_ok).to_frame(name='avg_realized_return'))
print('Long-only strategy average returns by signal:')
display(_signal_avg_returns(long_only_ok).to_frame(name='avg_realized_return'))
